# 08｜用简化 YOLO 串起完整目标检测流程

前七课分别学习了边界框、IoU、分类与回归、预测槽位、目标分配、NMS、Precision、Recall、AP 和 mAP。现在先不增加更多零件，而是把这些零件装进同一台机器。

这一课只跟着一张图片，回答一个完整问题：

**一张图片进入简化 YOLO 后，怎样变成最终的类别与边界框；训练时模型又怎样知道自己哪里做错了？**

> 这里讲的是 YOLO 类单阶段检测器的共同思路，不对应某一个具体版本。不同 YOLO 版本在网络结构、框的表示和目标分配规则上会有差异。

## 一、先看完整故事，再逐段拆开

假设图片中有一只狗和一只猫。一次推理大致经历：

$$
\text{图片}
\longrightarrow \text{提取图像特征}
\longrightarrow \text{许多预测单位同时给出候选结果}
\longrightarrow \text{解码成类别、分数和边界框}
\longrightarrow \text{分数筛选}
\longrightarrow \text{NMS 删除重复框}
\longrightarrow \text{最终检测结果}
$$

训练时会多出真实标注、目标分配、损失计算和反向传播；评价时会多出 TP、FP、FN、PR 曲线、AP 与 mAP。

所以 YOLO 并不是一组与前面无关的新概念，而是把已经学过的概念按照固定顺序运行起来。

## 二、第一步：图片先变成特征图

原始图片只是像素。模型不能直接从某个像素读出“这里有狗”，所以先让卷积网络逐层提取特征：

$$
\text{像素}
\longrightarrow \text{边缘与纹理}
\longrightarrow \text{局部形状}
\longrightarrow \text{物体语义}
$$

负责提取这些特征的主体通常叫作 **Backbone，也就是骨干网络**。它的输出不再是一张普通 RGB 图片，而是若干张特征图。

如果只看一张简化特征图，可以把形状写成：

$$
D\times H\times W
$$

$H\times W$ 表示特征图上的空间位置，$D$ 表示每个位置保存的特征数量。

### 特征图上的位置和原图位置有没有关系

有关系，但不是简单地把原图切成互不相干的小方块。

特征图左上方的位置通常更关注原图左上区域，右下方的位置通常更关注原图右下区域。不过，经过多层卷积以后，一个位置看到的范围会逐渐扩大，它保存的向量可能同时包含目标本身和周围环境的信息。这个能够影响该位置的原图范围叫作感受野。

因此可以这样理解：

- 特征图位置提供大致的空间落点；
- 该位置的特征向量提供判断物体所需的信息；
- 它不是只看一个像素，也不是只能预测与格子一样大的框。

## 三、第二步：特征图上放置许多预测单位

假设教学用特征图只有 $4\times4$ 个位置，并且每个位置只放一个预测单位，那么一共会有：

$$
4\times4=16
$$

个预测单位。它们会同时查看各自位置的特征向量，然后各自提出一条候选检测结果。

可以把预测单位理解成模型输出表中预先准备好的固定座位：

- 图片进来以前，16 个座位就已经存在；
- 图片进来以后，每个座位都得到一个由神经网络算出的特征向量；
- 预测头再把这个向量翻译成类别、目标性和边界框。

这里的“预测单位”和此前笔记中的“预测槽位”表达的是同一类抽象概念：**模型预先准备的一个输出位置。**

### YOLO 的预测单位为什么和位置有关

简化 YOLO 的预测单位依附在特征图位置上，因此它天然带有空间位置。假设狗的中心落在第 2 行、第 3 列附近，目标分配规则就可能让这个位置附近的某个预测单位负责狗。

但是“负责位置在这里”不表示边界框只能待在这个格子里。预测单位还会输出框的中心偏移和宽高，因此最终框可以向四周伸展，覆盖一整只大狗。

一句话记忆：

**特征图位置决定从哪里出发提出预测，边界框回归决定最终框覆盖多大范围。**

## 四、第三步：预测头把特征翻译成检测结果

骨干网络产生的是特征，不能直接拿来画框。预测头是网络末端的一小组可学习层，负责把每个预测单位的特征向量翻译成我们需要的数字。

对位置 $(i,j)$ 的特征向量 $z_{ij}$，简化预测头可以表示为：

$$
z_{ij}
\xrightarrow{\text{预测头}}
(o_{ij},\ p_{ij}^{(1)},\ldots,p_{ij}^{(C)},\ x_{ij},y_{ij},w_{ij},h_{ij})
$$

这些输出分别回答：

- $o_{ij}$：这里像不像存在一个物体，也叫目标性；
- $p_{ij}^{(1)},\ldots,p_{ij}^{(C)}$：如果有物体，它属于各类别的可能性；
- $x_{ij},y_{ij},w_{ij},h_{ij}$：边界框在哪里、有多大。

预测头不是把加工后的向量再放回槽位。更准确地说，槽位就是输出表中的位置；预测头在这个位置上直接产生最终预测数值。

### 预测单位里面的向量从哪里来

现在可以把此前的疑问完整回答了：

$$
\text{图片像素}
\longrightarrow \text{骨干网络提取特征}
\longrightarrow \text{特征图位置上的向量 }z_{ij}
\longrightarrow \text{预测头加工}
\longrightarrow \text{目标性、类别和边界框}
$$

所以“向量从神经网络中来，又经过神经网络加工变成有用信息”这个理解是对的。还可以再精确一点：前面的网络负责形成图像特征向量，末端预测头负责把特征向量变成检测任务需要的数字。

## 五、简化输出张量的形状

假设：

- batch 大小为 $B$；
- 特征图大小为 $H\times W$；
- 每个位置只有一个预测单位；
- 数据集有 $C$ 个物体类别；
- 每个单位输出 1 个目标性、$C$ 个类别值和 4 个框坐标。

那么简化输出形状是：

$$
B\times H\times W\times(1+C+4)
$$

也就是：

$$
B\times H\times W\times(5+C)
$$

真实 YOLO 往往使用多个尺度，并可能在一个位置安排多个预测单位。那时会多一个预测单位数量 $A$，形状可以抽象为 $B\times H\times W\times A\times(5+C)$。这一课只需先守住“每个单位输出一组检测数字”这条主线。

## 六、到这里，网络只是产生了许多原始候选

对于 $4\times4$ 的教学特征图，16 个预测单位都会输出结果。即使图片中只有狗和猫两个真实目标，网络仍然会先产生 16 条候选。

这不表示模型认为图中一定有 16 个物体。大多数单位应该给出较低目标性，表示自己更像背景；少数单位应该给出较高目标性，并预测狗或猫的类别与边界框。

至此我们只完成了前向传播：

$$
\text{输入图片}
\longrightarrow \text{网络}
\longrightarrow \text{固定数量的原始候选预测}
$$

接下来必须分开讨论训练和推理，因为它们拿到这些候选以后做的事情不同。

## 七、训练路线：先做目标分配

训练图片附带真实标注，例如：

- 真实目标 1：狗及其真实框；
- 真实目标 2：猫及其真实框。

但模型有 16 个预测单位，真实目标只有 2 个。目标分配要建立一张责任表：哪个预测单位负责狗，哪个负责猫，哪些负责学习背景。

$$
\text{真实类别与真实框}
\xrightarrow{\text{目标分配规则}}
\text{每个预测单位的监督身份与学习目标}
$$

分到真实目标的单位成为正样本；没有分到真实目标的单位通常成为负样本或在某些规则下被忽略。具体怎样选负责单位因 YOLO 版本而异，这一课不学习某个版本的细则。

### 目标分配不是预测头之后的另一个神经网络

目标分配通常是一套训练规则，不是需要学习参数的预测层。它会读取真实标注，并可能参考预测单位的位置、尺度、候选框或当前预测结果，然后产生监督关系。

网络前向传播和目标分配的关系可以画成两条汇合的路线：

$$
\begin{aligned}
\text{图片}&\longrightarrow\text{网络输出的预测}\\
\text{真实标注}&\longrightarrow\text{目标分配产生的监督目标}
\end{aligned}
\quad\Longrightarrow\quad
\text{计算损失}
$$

因此，目标分配的主要任务不是继续加工图像，而是告诉损失函数：每个预测单位应该和谁比较。

## 八、正样本和负样本分别学习什么

假设一个预测单位被分配给狗，它是正样本。训练会要求它：

- 目标性变高，表示这里有物体；
- 狗类别的分数变高；
- 预测框逐渐靠近狗的真实框。

没有分配到目标的单位是负样本。训练通常主要要求它把目标性压低，表示这里更像背景。因为背景没有真实边界框，所以一般不要求负样本把框回归到某个所谓的背景框。

这也回答了“模型什么时候学习物体，什么时候学习背景”：

- 正样本推动模型学习物体类别和位置；
- 负样本推动模型识别哪些候选不应该被当成物体；
- 它们在同一个 batch、同一次损失计算中一起训练模型。

## 九、三个损失怎样合作

为了教学，可以把 YOLO 损失抽象成三部分：

$$
\mathcal{L}
=\lambda_{\mathrm{obj}}\mathcal{L}_{\mathrm{obj}}
+\lambda_{\mathrm{cls}}\mathcal{L}_{\mathrm{cls}}
+\lambda_{\mathrm{box}}\mathcal{L}_{\mathrm{box}}
$$

它们分别纠正：

- 目标性损失：这里究竟有物体还是更像背景；
- 分类损失：如果有物体，它是什么类别；
- 边界框损失：框的位置和大小是否准确。

损失计算完以后，通过反向传播同时修改骨干网络和预测头的参数。下一次看到相似图像时，特征与输出都有机会变得更合理。

### 一次训练迭代的完整路线

现在把训练过程完整串起来：

$$
\begin{aligned}
\text{图片}
&\longrightarrow \text{骨干网络与特征融合}
\longrightarrow \text{预测头}
\longrightarrow \text{大量原始预测}\\
\text{真实标注}
&\longrightarrow \text{目标分配}
\longrightarrow \text{正样本、负样本及其监督目标}\\
\text{原始预测与监督目标}
&\longrightarrow \text{计算目标性、分类和框损失}\\
&\longrightarrow \text{反向传播与参数更新}
\end{aligned}
$$

训练阶段的最终目的不是输出漂亮的检测框，而是用损失产生梯度，让下一次预测更好。

## 十、推理路线：没有真实标注，也不计算损失

模型训练好以后，输入一张新图片。此时没有真实框可供目标分配，也没有答案可以计算损失。模型只能依靠学到的参数产生候选预测。

简化推理流程是：

$$
\text{新图片}
\longrightarrow \text{特征图}
\longrightarrow \text{所有预测单位的原始输出}
\longrightarrow \text{解码边界框与类别分数}
\longrightarrow \text{后处理}
\longrightarrow \text{最终结果}
$$

很多 YOLO 风格的解释会把目标性和类别可能性结合成某类别的最终分数：

$$
s_c\approx P(\text{有物体})\times P(c\mid\text{有物体})
$$

这是帮助理解的常见写法，具体版本的输出和分数定义可能不同。

## 十一、为什么推理时还需要后处理

密集预测单位会产生大量候选，其中通常包含两类不需要的结果：

1. 分数很低的背景候选；
2. 多个单位对同一个真实物体给出的重复框。

所以后处理通常分两步：

$$
\text{大量候选框}
\xrightarrow{\text{分数阈值}}
\text{删除明显不可信的框}
\xrightarrow{\text{按类别 NMS}}
\text{删除同一物体的重复框}
$$

分数筛选主要处理“太不可信”，NMS 主要处理“太重复”。最后留下的才是展示给用户的检测结果。

### 为什么训练已经分配了目标，推理仍可能重复

目标分配是在训练时安排责任，但神经网络并不会被硬性锁住。相邻位置可能看到相似特征，也可能同时认为自己发现了狗。

尤其在密集检测器中，一个真实目标周围往往存在多个有能力提出预测的单位。训练规则、数据和损失会影响它们的行为，却不能保证推理时永远只输出一个框。

因此传统 YOLO 风格流程允许先多提候选，再使用 NMS 选出代表框。

## 十二、评价路线：mAP 不参与生成预测

得到最终预测以后，如果当前是在验证集或测试集上，还要把预测与真实标注进行比较：

$$
\text{最终预测与真实标注}
\longrightarrow \text{按类别和评价 IoU 阈值匹配}
\longrightarrow TP,FP,FN
\longrightarrow \text{PR 曲线}
\longrightarrow AP
\longrightarrow mAP
$$

mAP 是评价结果，不是预测头的一部分，也不是模型用来删除框的规则。

训练时可以定期在验证集上计算 mAP，观察模型整体检测能力是否提高；但一次普通推理并不需要先计算 mAP 才能输出结果。

## 十三、把训练、推理和评价彻底分开

| 阶段 | 有真实标注吗 | 做目标分配吗 | 计算损失并更新参数吗 | 做分数筛选和 NMS 吗 | 可能计算 mAP 吗 |
|---|---|---|---|---|---|
| 训练 | 有 | 有 | 有 | 通常不是训练主线 | 可在验证时另算 |
| 普通推理 | 没有 | 没有 | 没有 | 有 | 不需要 |
| 验证或测试 | 有，但只用于评价 | 不做训练目标分配 | 不更新参数 | 有 | 有 |

最容易混淆的是：验证集虽然有真实标注，但模型生成预测时并不能偷看答案。真实标注只在预测完成后用于匹配和计算指标。

## 十四、完整跟踪图片中的一只狗

现在只跟踪狗，观察它在整条流程中怎样出现：

1. 原图中有一只狗，训练标注给出狗类别和真实框；
2. 骨干网络把像素变成特征图，狗附近的位置形成包含狗特征的向量；
3. 目标分配选择一个或一些预测单位负责这只狗；
4. 预测头为这些单位输出目标性、狗类别分数和边界框；
5. 损失比较预测与监督目标，反向传播让网络逐渐改进；
6. 推理时，多个单位可能都给狗输出高分框；
7. 分数筛选删除低分候选，NMS 从重叠狗框中保留代表框；
8. 评价时，代表框与狗的真实框匹配，可能成为 TP；
9. 整个验证集的 TP、FP 排序最终形成 PR 曲线、AP 和 mAP。

这九步就是前七课知识在一个目标身上的完整旅程。

## 十五、真实 YOLO 为什么还会使用多尺度特征

单张 $4\times4$ 特征图只是教学简化。真实图片中的物体大小差异很大：远处的小猫可能只占很少像素，近处的大狗可能占据大半张图。

因此真实 YOLO 通常会让不同分辨率的特征图共同参与检测：

- 分辨率较高的特征图保留更多空间细节，更适合小目标；
- 分辨率较低、语义更强的特征图更适合较大目标；
- Neck，也就是特征融合部分，会把不同层级的信息重新组合；
- 检测头在多个尺度上产生候选预测。

无论具体版本怎样改变，核心仍然是：特征图提供信息，预测单位提出候选，预测头输出类别与框。

## 十六、这一课的简化边界

为了先建立稳定主线，本课有意简化了以下内容：

- 没有区分 YOLOv1、YOLOv3、YOLOX、YOLOv8 等版本；
- 没有展开 Anchor-based 与 Anchor-free 的差别；
- 没有学习某个版本的坐标解码公式；
- 没有学习动态目标分配、正负样本筛选的具体算法；
- 没有进入网络代码、数据增强和训练配置。

这些差异会改变实现细节，但不会推翻本课的完整主线。现在最重要的是能够说清每个环节为什么存在，以及训练、推理、评价之间怎样衔接。

## 十七、常见误区

### 误区 1：特征图上的一个位置只看到原图中的一个小格子

不是。它与原图位置有关，但经过多层网络后会拥有一定感受野，能够包含周围上下文。

### 误区 2：预测槽位是一个容器，预测头把答案放进去以后还会继续加工

不必这样想。槽位只是固定输出位置；该位置的特征经过预测头后，直接形成这一槽位的检测输出。

### 误区 3：负样本什么都不学

负样本会学习降低目标性，帮助模型识别背景和错误候选。

### 误区 4：目标分配在推理时帮助模型寻找物体

不是。目标分配依赖训练标注，主要在训练时建立监督关系；新图片没有真实标注。

### 误区 5：mAP 是网络输出的一部分

不是。mAP 是用预测结果和真实标注计算出来的评价指标。

## 十八、自我检查

尝试不看笔记回答：

1. 图片为什么不能直接跳过特征提取，让每个预测单位只看一个像素？
2. YOLO 的预测单位和特征图位置有什么关系？
3. 预测头对一个特征向量输出哪三类信息？
4. 目标分配的输入和输出分别是什么？
5. 正样本和负样本分别推动模型学习什么？
6. 为什么推理时没有目标分配和损失？
7. 分数筛选与 NMS 分别解决什么问题？
8. mAP 位于训练、推理还是评价路线？

### 参考答案

1. 单个像素几乎没有物体语义；网络需要从像素逐层提取包含形状、语义和上下文的特征。
2. 简化 YOLO 的预测单位依附在特征图位置上，因此有空间落点，但输出框可以覆盖远大于该位置对应格子的范围。
3. 目标性、类别分数和边界框坐标。
4. 输入至少包含真实标注和候选预测单位信息；输出是单位与真实目标之间的责任关系以及监督目标。
5. 正样本学习发现物体、判断类别和定位；负样本学习压低背景候选的目标性。
6. 新图片没有真实标注，模型只能前向生成预测，不能建立训练监督或计算训练损失。
7. 分数筛选删除明显不可信的候选，NMS 删除同一物体的重复框。
8. mAP 属于评价路线，用预测结果与真实标注计算。

## 十九、本课只需要记住一条总主线

### 训练

$$
\text{图片产生预测}
+\text{真实标注产生监督关系}
\longrightarrow\text{损失}
\longrightarrow\text{更新网络}
$$

### 推理

$$
\text{图片}
\longrightarrow\text{大量候选}
\longrightarrow\text{分数筛选与 NMS}
\longrightarrow\text{最终检测结果}
$$

### 评价

$$
\text{最终检测结果与真实标注}
\longrightarrow TP,FP,FN
\longrightarrow\text{PR 曲线、AP 与 mAP}
$$

能把这三条路线讲清楚，就已经真正把目标检测共同基础连成了整体。

## 下一课：从 YOLO 走回 DETR

传统密集检测流程已经能够工作，但它留下了一个值得追问的问题：

**模型为什么要先产生大量重复候选，再依靠 NMS 删除？能不能直接输出一组不重复的最终结果？**

下一课将以这个问题为起点，对比两种思路：

$$
\begin{aligned}
\text{YOLO 风格：密集预测单位}&\longrightarrow\text{大量候选}\longrightarrow\text{NMS}\\
\text{DETR：Object Queries}&\longrightarrow\text{一组预测}\longrightarrow\text{一对一集合预测}
\end{aligned}
$$

这时再学习 DETR 的预测槽位、Object Queries 和匈牙利匹配，就能清楚看到它究竟改变了哪里，而不是只记住几个新名词。